# Week 2 — Task 2.3: Feature Engineering

Deterministic, memory-conscious predictor construction only. No split, fitted encoding, scaling, target-derived features or modelling.


## 1. Input validation


In [ ]:
from pathlib import Path
import json, sys
import pyarrow.parquet as pq
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.week2_feature_engineering import (DEFAULT_INPUT, DEFAULT_OUTPUT, DEFAULT_METRICS, validate_input, create_features, feature_dictionary)
source = pq.ParquetFile(DEFAULT_INPUT)
validate_input(source, ROOT/'reports'/'week2_preprocessing_statistics.json')


## 2. Feature-engineering goals

Retain all original data while adding compact calendar, building, weather and interaction predictors that are available without reading the target.


## 3. Time features

Calendar integers retain timestamp semantics. Sine/cosine pairs represent hour, weekday and month cycles without false endpoint distance.


## 4. Building features

Age is anchored to 2016 and clipped at zero; log area compresses scale; area per floor uses a denominator bounded below by one.


## 5. Weather features

Features include temperature/dew spread, Magnus relative humidity, circular wind direction, documented indicators, and 18 °C degree proxies.


## 6. Interaction features

Four bounded-order multiplicative terms connect building scale/vintage with temperature or degree demand. None uses target statistics.


In [ ]:
sample = source.read_row_group(0).slice(0, 10_000).to_pandas()
preview = create_features(sample)
preview.head()


## 7. Lag-feature decision

Deferred because prediction horizon and inference-time target history are unknown. Future lags require chronological (`building_id`, `meter`) grouping and a shift before rolling.


## 8. Run and feature validation

Run the CLI from the project root to reproduce the full artifact: `python scripts/week2_feature_engineering.py`. The script independently reloads all row groups and validates keys, values, ranges, target preservation and input fingerprint.


In [ ]:
metrics = json.loads(DEFAULT_METRICS.read_text(encoding='utf-8'))
metrics['validation']


## 9. Memory impact


In [ ]:
{k: metrics[k] for k in ['input','output','added_column_count','processing_duration_seconds']}


## 10. Final feature inventory


In [ ]:
import pandas as pd
pd.DataFrame(metrics['feature_dictionary'])


## 11. Readiness for model splitting

The timestamp and grouping identifiers are retained. The dataset is ready for a separate chronological splitting task; no split or model preparation is performed here.
